# Open-Loop Rollout Diagnostic

For a Dreamer v1 world model, this notebook tests whether the model can predict
future observations using only the prior — i.e., without seeing observations
during the rollout. This is the diagnostic that matters for whether imagination-based
policy training will work, since the actor's training trajectory is a prior rollout.

**What we check:**
1. Closed-loop reconstruction is sharp (sanity).
2. Open-loop predictions track reality across `horizon` steps.
3. Open-loop error is bounded — doesn't explode.
4. Open-loop error beats a zero-dynamics baseline ("predict no change").
5. Per-dim error highlights which observation components the model handles well.
6. Reward prediction is accurate over the horizon (what the actor actually relies on).

In [ ]:
# Parameters — papermill injects overrides after this cell.
checkpoint_path = "checkpoints/default/latest.safetensors"
data_path       = "data/generated-pushes/"
data_format     = "rerun"   # "hdf5" or "rerun"
num_episodes    = 20
burn_in         = 5
horizon         = 15
seed            = 0


## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np

from chuck_dreamer.dreamer import build_model, rollout
from chuck_dreamer.dreamer.mlx_model import load_config_from_checkpoint
from chuck_dreamer.sim import PushingEnv
from chuck_dreamer.training.episode_dataset import EpisodeDataset
from chuck_dreamer.training.episode_processor import processor_for

np.random.seed(seed)
mx.random.seed(seed)

print(f"checkpoint:    {checkpoint_path}")
print(f"data:          {data_path} ({data_format})")
print(f"num_episodes:  {num_episodes}")
print(f"burn_in:       {burn_in}")
print(f"horizon:       {horizon}")

## Load model and evaluation episodes

Checkpoints written by ``Trainer`` embed the training config in their safetensors
metadata block. We recover that config here so the rebuilt model has the right
shapes for whatever ``env.obs_mode`` / ``env.act_mode`` combination the checkpoint
was trained with.

In [ ]:
config = load_config_from_checkpoint(checkpoint_path)
if config is None:
  raise RuntimeError(
    f"checkpoint {checkpoint_path!r} has no embedded config metadata; "
    "re-train or re-save with a current Trainer to embed the config."
  )

# Build env to derive obs/action shapes (matches Trainer.__init__).
env        = PushingEnv(config)
obs_shape  = env.model_obs_shape
action_dim = int(env.action_space.shape[0])
obs_mode   = env.obs_mode

model = build_model(config, obs_shape=obs_shape, action_dim=action_dim)
# Switch to eval mode: load weights without rebuilding the optimizer.
model.training = False
model.load(checkpoint_path)

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"obs_shape={obs_shape}  action_dim={action_dim}  obs_mode={obs_mode}  act_mode={env.act_mode}")
print(f"RSSM dims: stoch={model.rssm.stoch_dim}, deter={model.rssm.deter_dim}")

In [ ]:
# Load evaluation episodes through the same processor the replay buffer uses.
processor = processor_for(config)
min_len   = burn_in + horizon

episodes: list[dict] = []
for raw_episode in EpisodeDataset(data_path, format=data_format).stream():
  ep = processor(raw_episode.data)
  if ep["action"].shape[0] >= min_len:
    episodes.append(ep)
  if len(episodes) >= num_episodes:
    break

print(f"Using {len(episodes)} episodes (>= {min_len} steps)")

## Rollout helper

We use :func:`~chuck_dreamer.dreamer.eval_split_rollout` to run two
continuations from a shared posterior burn-in: a posterior-only tail
(teacher-forced) and a prior-only tail (open-loop). The posterior tail is
the upper bound on what the model can reconstruct given perfect future
observations; the prior tail is what imagination-based policy training
actually consumes. We report metrics for both.

The RSSM advances ``state_t = obs_step(state_{t-1}, action_{t-1}, embed_t)``
where ``embed_t`` is the embedding of the obs *after* ``action_{t-1}``.
The replay buffer stores ``obs[t]`` *before* ``action[t]``, so we drop the
first embed when feeding ``eval_split_rollout`` and align predictions
with ``obs[1:]``.

In [ ]:
def _batch_obs(obs):
  """Wrap one episode's obs in a leading batch axis.

  Handles flat ndarray observations (state / image) as well as the
  ``{"image": ..., "proprio": ...}`` dict produced by the image_proprio
  processor.
  """
  if isinstance(obs, dict):
    return {k: mx.array(v[None]) for k, v in obs.items()}
  return mx.array(obs[None])


def _to_np(pred, T):
  """Take a model output (ndarray or dict), drop the batch axis, slice to T."""
  if isinstance(pred, dict):
    return {k: np.asarray(v[0])[:T] for k, v in pred.items()}
  return np.asarray(pred[0])[:T]


def _slice_episode_obs(obs, lo, hi):
  """Slice an episode's obs window, handling ndarray or dict."""
  if isinstance(obs, dict):
    return {k: v[lo:hi] for k, v in obs.items()}
  return obs[lo:hi]


def open_loop_rollout(model, obs, actions, burn_in, horizon):
  """Run one episode through the model twice: full-posterior and burn-in+prior.

  obs:     ndarray ``(T+1, ...)`` or dict of such — obs[t] is BEFORE action[t].
  actions: ``(T, action_dim)`` ndarray.
  Returns predicted obs and rewards for both rollouts across the full
  ``burn_in + horizon`` window.
  """

  T_total = burn_in + horizon
  obs_b   = _batch_obs(obs)
  act_b   = mx.array(actions[:T_total][None])         # (1, T_total, A)
  embeds  = model.encode(obs_b)[:, 1:T_total + 1]     # embed of obs AFTER action_t

  # Full posterior across the whole window — upper bound on reconstruction.
  post = rollout(
    model, init_state=model.initial_state(1), horizon=T_total,
    mode="posterior", actions=act_b, embeds=embeds, sample=False,
    decode=True, predict_reward=True,
  )
  # Burn-in posterior, then open-loop prior continuation — what imagination uses.
  prior_split = rollout(
    model, init_state=model.initial_state(1), horizon=T_total,
    mode="prior", burn_in=burn_in,
    actions=act_b, embeds=embeds, sample=False,
    decode=True, predict_reward=True,
  )

  return {
    "obs_pred_post":  _to_np(post.obs_pred, T_total),
    "rew_pred_post":  np.asarray(post.rewards[0])[:T_total],
    "obs_pred_prior": _to_np(prior_split.obs_pred, T_total),
    "rew_pred_prior": np.asarray(prior_split.rewards[0])[:T_total],
  }

## Run rollouts

In [ ]:
results = []
for ep in episodes:
  T_total = burn_in + horizon
  if ep["action"].shape[0] < T_total:
    continue
  out = open_loop_rollout(model, ep["obs"], ep["action"], burn_in, horizon)
  results.append({
    # observation AFTER each action; matches obs_pred timing
    "obs":            _slice_episode_obs(ep["obs"], 1, T_total + 1),
    "obs_pred_prior": out["obs_pred_prior"],
    "obs_pred_post":  out["obs_pred_post"],
    "reward":         ep["reward"][:T_total],
    "rew_pred_prior": out["rew_pred_prior"],
    "rew_pred_post":  out["rew_pred_post"],
  })
print(f"Computed rollouts for {len(results)} episodes")

## Diagnostic 1 — Open-loop error vs horizon

Per-step L2 error between predicted and true observation, averaged over episodes.
The vertical line marks the burn-in boundary; before it the model sees observations,
after it the model is in open-loop mode.

We compare against a **zero-dynamics baseline**: "predict that the obs stays equal
to the last observed obs forever." For Lipschitz-continuous physical systems,
consecutive observations are close, so this is a decent baseline at short horizons
and a bad one at longer horizons. Your model should beat it after a few steps.

In [ ]:
def _normalize_image(img):
  """Match the encoder's pixel normalization: uint8 [0,255] -> float [-0.5, 0.5]."""
  if img.dtype == np.uint8:
    return img.astype(np.float32) / 255.0 - 0.5
  return img.astype(np.float32, copy=False)


def per_step_l2_state(pred, true):
  return np.linalg.norm(pred - true, axis=-1)


def per_step_l2_image(pred, true):
  diff = (pred - _normalize_image(true)).reshape(pred.shape[0], -1)
  return np.linalg.norm(diff, axis=-1)


def per_step_l2(pred, true, mode):
  if mode == "state":
    return per_step_l2_state(pred, true)
  if mode == "image":
    return per_step_l2_image(pred, true)
  if mode == "image_proprio":
    img = per_step_l2_image(pred["image"], true["image"])
    pro = per_step_l2_state(pred["proprio"], true["proprio"])
    return img + pro
  raise ValueError(f"unknown obs_mode={mode!r}")


def zero_dynamics_pred(true_window, mode):
  """Predict 'obs stays at the last burn-in obs' for the whole window."""
  if mode == "image_proprio":
    last = {k: v[burn_in - 1] for k, v in true_window.items()}
    T = next(iter(true_window.values())).shape[0]
    out = {}
    for k, v in last.items():
      tiled = np.tile(v[None], (T, *([1] * v.ndim)))
      out[k] = _normalize_image(tiled) if k == "image" else tiled
    return out
  last = true_window[burn_in - 1] if burn_in > 0 else true_window[0]
  T = true_window.shape[0]
  tiled = np.tile(last[None], (T, *([1] * last.ndim)))
  if mode == "image":
    tiled = _normalize_image(tiled)
  return tiled


T_max = burn_in + horizon
errs_prior = np.full((len(results), T_max), np.nan)
errs_post  = np.full((len(results), T_max), np.nan)
errs_zero  = np.full((len(results), T_max), np.nan)

for i, r in enumerate(results):
  errs_prior[i] = per_step_l2(r["obs_pred_prior"], r["obs"], obs_mode)
  errs_post [i] = per_step_l2(r["obs_pred_post"],  r["obs"], obs_mode)
  if burn_in > 0:
    zero_pred = zero_dynamics_pred(r["obs"], obs_mode)
    errs_zero[i] = per_step_l2(zero_pred, r["obs"], obs_mode)

mean_prior = np.nanmean(errs_prior, axis=0)
mean_post  = np.nanmean(errs_post,  axis=0)
mean_zero  = np.nanmean(errs_zero,  axis=0)
std_prior  = np.nanstd(errs_prior,  axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(T_max)
ax.plot(xs, mean_prior, label="prior (open-loop)", color="C0", linewidth=2)
ax.fill_between(xs, mean_prior - std_prior, mean_prior + std_prior, alpha=0.2, color="C0")
ax.plot(xs, mean_post, label="posterior (teacher-forced)", color="C1", linewidth=2)
ax.plot(xs, mean_zero, label="zero-dynamics baseline", color="C3",
        linestyle="--", linewidth=1.5)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("step (action index)")
ax.set_ylabel(f"obs L2 error ({obs_mode})")
ax.set_title("Rollout error vs horizon: prior (open-loop) vs posterior (teacher-forced)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Diagnostic 2 — Per-dimension / per-component open-loop error

For ``obs_mode="state"`` the obs vector is a fixed concatenation produced by
``StateVectorProcessor``: ``ee_pos (3) + ee_quat (4) + object_xy (2) + joint_qpos (n_joints)``.
A per-dim bar chart highlights which obs components the model handles well — joint
angles are often easy (they follow nearly directly from actions), object position is
hard (it requires modeling contact dynamics).

For image modes, per-pixel error isn't useful as a bar chart. Instead we render
real vs predicted frames side-by-side at a few open-loop timesteps so divergence
in shape/colour/position is visible. For ``image_proprio`` we additionally show
the per-dim error on the proprio vector.

In [ ]:
def _state_obs_layout(obs_dim):
  """Group / label scheme for the state obs vector.

  Layout matches ``StateVectorProcessor``: ee_pos (3) + ee_quat (4) +
  object_xy (2) + joint_qpos (n).
  """
  n_joints = obs_dim - 9
  groups = {
    "ee_pos":     list(range(0, 3)),
    "ee_quat":    list(range(3, 7)),
    "object_xy":  list(range(7, 9)),
    "joint_qpos": list(range(9, obs_dim)),
  }
  labels = (
    ["ee_x", "ee_y", "ee_z"]
    + ["ee_qw", "ee_qx", "ee_qy", "ee_qz"]
    + ["obj_x", "obj_y"]
    + [f"q_{i}" for i in range(n_joints)]
  )
  assert len(labels) == obs_dim, (len(labels), obs_dim)
  return groups, labels


def _proprio_layout(proprio_dim):
  """Group / label scheme for image_proprio's proprio vector.

  Layout matches ``ImageProprioProcessor``: ee_pos (3) + ee_quat (4) +
  joint_qpos (n) — no object_xy.
  """
  n_joints = proprio_dim - 7
  groups = {
    "ee_pos":     list(range(0, 3)),
    "ee_quat":    list(range(3, 7)),
    "joint_qpos": list(range(7, proprio_dim)),
  }
  labels = (
    ["ee_x", "ee_y", "ee_z"]
    + ["ee_qw", "ee_qx", "ee_qy", "ee_qz"]
    + [f"q_{i}" for i in range(n_joints)]
  )
  assert len(labels) == proprio_dim, (len(labels), proprio_dim)
  return groups, labels


def _plot_per_dim(per_dim, groups, labels, title):
  colors = []
  for i in range(len(per_dim)):
    for g_idx, (_, dims) in enumerate(groups.items()):
      if i in dims:
        colors.append(f"C{g_idx}")
        break
    else:
      colors.append("gray")

  fig, ax = plt.subplots(figsize=(10, 4))
  ax.bar(np.arange(len(per_dim)), per_dim, color=colors)
  ax.set_xticks(np.arange(len(per_dim)))
  ax.set_xticklabels(labels, rotation=45, ha="right")
  ax.set_ylabel("mean |error| in open-loop region")
  ax.set_title(title)
  for g_idx, name in enumerate(groups):
    ax.bar([], [], color=f"C{g_idx}", label=name)
  ax.legend(); ax.grid(alpha=0.3, axis="y")
  plt.tight_layout(); plt.show()


def _per_dim_state(results_, dim_getter_pred, dim_getter_true, dim):
  """Average |pred - true| across the open-loop region, per dim. Uses the
  prior (open-loop) prediction — that's the regime the actor sees."""
  per_dim = np.zeros(dim)
  counts  = np.zeros(dim)
  for r in results_:
    T = r["reward"].shape[0]
    pred_arr = dim_getter_pred(r)
    true_arr = dim_getter_true(r)
    for t in range(burn_in, min(burn_in + horizon, T)):
      per_dim += np.abs(pred_arr[t] - true_arr[t])
      counts  += 1
  return per_dim / np.maximum(counts, 1)


def _to_uint8_image(img):
  """Map predicted [-0.5, 0.5] floats (or true uint8) back to displayable uint8."""
  if img.dtype == np.uint8:
    return img
  return np.clip((img + 0.5) * 255.0, 0, 255).astype(np.uint8)


def _plot_image_strip(results_, get_image_pred, get_image_true, title):
  r = results_[0]
  T = r["reward"].shape[0]
  ts = [t for t in (burn_in, burn_in + 2, burn_in + 5, burn_in + 10, burn_in + horizon - 1)
        if t < T]
  if not ts:
    print(f"episode 0 too short ({T} steps) for image strip at burn_in={burn_in}")
    return

  pred_imgs = get_image_pred(r)
  true_imgs = get_image_true(r)
  fig, axes = plt.subplots(2, len(ts), figsize=(3 * len(ts), 6))
  axes = np.atleast_2d(axes)
  for col, t in enumerate(ts):
    axes[0, col].imshow(_to_uint8_image(true_imgs[t]))
    axes[0, col].set_title(f"real t={t}")
    axes[0, col].axis("off")
    axes[1, col].imshow(_to_uint8_image(pred_imgs[t]))
    axes[1, col].set_title(f"pred t={t}")
    axes[1, col].axis("off")
  fig.suptitle(title)
  plt.tight_layout(); plt.show()


if obs_mode == "state":
  obs_dim = results[0]["obs"].shape[-1]
  groups, labels = _state_obs_layout(obs_dim)
  per_dim = _per_dim_state(
    results,
    dim_getter_pred=lambda r: r["obs_pred_prior"],
    dim_getter_true=lambda r: r["obs"],
    dim=obs_dim,
  )
  _plot_per_dim(per_dim, groups, labels,
                f"Per-dim prediction error (prior tail), steps {burn_in}–{burn_in+horizon-1}")

elif obs_mode == "image":
  _plot_image_strip(
    results,
    get_image_pred=lambda r: r["obs_pred_prior"],
    get_image_true=lambda r: r["obs"],
    title="Episode 0 — real vs prior-predicted frames (top: real, bottom: predicted)",
  )

elif obs_mode == "image_proprio":
  _plot_image_strip(
    results,
    get_image_pred=lambda r: r["obs_pred_prior"]["image"],
    get_image_true=lambda r: r["obs"]["image"],
    title="Episode 0 — real vs prior-predicted frames (top: real, bottom: predicted)",
  )
  proprio_dim = results[0]["obs"]["proprio"].shape[-1]
  groups, labels = _proprio_layout(proprio_dim)
  per_dim = _per_dim_state(
    results,
    dim_getter_pred=lambda r: r["obs_pred_prior"]["proprio"],
    dim_getter_true=lambda r: r["obs"]["proprio"],
    dim=proprio_dim,
  )
  _plot_per_dim(per_dim, groups, labels,
                f"Per-dim proprio error (prior tail), steps {burn_in}–{burn_in+horizon-1}")

else:
  raise ValueError(f"unknown obs_mode={obs_mode!r}")

## Diagnostic 3 — Reward prediction error

The actor's training relies on the reward predictor's output during imagination.
If reward prediction degrades fast in open-loop, the actor's gradient signal is
wrong even if obs reconstruction looks fine. This is the metric most directly
predictive of whether the actor will train successfully.

In [ ]:
rew_errs_prior = np.full((len(results), T_max), np.nan)
rew_errs_post  = np.full((len(results), T_max), np.nan)
all_rewards = np.concatenate([r["reward"] for r in results])
mean_reward = float(all_rewards.mean()) if all_rewards.size else 0.0

err_array = np.full((len(results), T_max), np.nan)
for i, r in enumerate(results):
  rew_errs_prior[i] = np.abs(r["rew_pred_prior"] - r["reward"])
  rew_errs_post [i] = np.abs(r["rew_pred_post"]  - r["reward"])
  err_array[i] = np.abs(mean_reward - r["reward"])
mean_pred_err = np.nanmean(err_array, axis=0)

mean_prior_rew = np.nanmean(rew_errs_prior, axis=0)
mean_post_rew  = np.nanmean(rew_errs_post,  axis=0)
std_prior_rew  = np.nanstd(rew_errs_prior,  axis=0)

fig, ax = plt.subplots(figsize=(9, 4))
xs = np.arange(T_max)
ax.plot(xs, mean_prior_rew, label="prior reward error",     color="C2", linewidth=2)
ax.fill_between(xs, mean_prior_rew - std_prior_rew, mean_prior_rew + std_prior_rew,
                alpha=0.2, color="C2")
ax.plot(xs, mean_post_rew,  label="posterior reward error", color="C1", linewidth=2)
ax.plot(xs, mean_pred_err, label="mean-predictor baseline",
        color="C3", linestyle="--", linewidth=1.5)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("step")
ax.set_ylabel("|predicted - true| reward")
ax.set_title("Reward prediction error vs horizon")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Diagnostic 4 — Single-episode trajectory comparison

For ``state`` (and the proprio component of ``image_proprio``) we plot every dim of
obs across one episode, overlaying real vs predicted. The eye is good at spotting
when prediction visibly diverges. Use this as a qualitative sanity check; the
quantitative checks above are what you actually track over training.

For pure-image checkpoints there's no scalar trajectory to plot — Diagnostic 2's
image strip is the qualitative companion.

In [ ]:
def _plot_trajectory(true_arr, pred_arr, dim_labels, title):
  T, D = true_arr.shape
  ncols = 4
  nrows = (D + ncols - 1) // ncols
  fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 2.5 * nrows),
                           sharex=True)
  axes = np.atleast_2d(axes).flatten()
  xs = np.arange(T)
  for d in range(D):
    ax = axes[d]
    ax.plot(xs, true_arr[:, d], label="real", color="C0", linewidth=1.5)
    ax.plot(xs, pred_arr[:, d], label="pred (prior)", color="C1",
            linestyle="--", linewidth=1.5)
    ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5)
    ax.set_title(dim_labels[d], fontsize=9)
    ax.grid(alpha=0.3)
  for d in range(D, len(axes)):
    axes[d].axis("off")
  axes[0].legend(loc="upper left", fontsize=8)
  fig.suptitle(title)
  plt.tight_layout(); plt.show()


ep_idx = 0   # change to inspect different episodes
r = results[ep_idx]

if obs_mode == "state":
  obs_dim = r["obs"].shape[-1]
  _, labels = _state_obs_layout(obs_dim)
  _plot_trajectory(r["obs"], r["obs_pred_prior"], labels,
                   f"Episode {ep_idx}: real vs prior-predicted obs (dashed line = burn-in end)")

elif obs_mode == "image_proprio":
  proprio_dim = r["obs"]["proprio"].shape[-1]
  _, labels = _proprio_layout(proprio_dim)
  _plot_trajectory(r["obs"]["proprio"], r["obs_pred_prior"]["proprio"], labels,
                   f"Episode {ep_idx}: real vs prior-predicted proprio (dashed line = burn-in end)")

elif obs_mode == "image":
  print("obs_mode=image: see Diagnostic 2's image strip for qualitative comparison.")

else:
  raise ValueError(f"unknown obs_mode={obs_mode!r}")

## Summary

The numbers below are the headline metrics. Track these across checkpoints to see
whether world model quality is improving over training.

In [ ]:
def _at(arr, idx):
  return float(arr[idx]) if 0 <= idx < arr.shape[0] else None

summary = {
  "checkpoint":                 str(checkpoint_path),
  "obs_mode":                   obs_mode,
  "act_mode":                   env.act_mode,
  "num_episodes":               len(results),
  "burn_in":                    burn_in,
  "horizon":                    horizon,
  "prior_obs_err_at_h1":        _at(mean_prior, burn_in),
  "prior_obs_err_at_h5":        _at(mean_prior, burn_in + 4),
  "prior_obs_err_at_h15":       _at(mean_prior, burn_in + 14),
  "post_obs_err_at_h1":         _at(mean_post,  burn_in),
  "post_obs_err_at_h5":         _at(mean_post,  burn_in + 4),
  "post_obs_err_at_h15":        _at(mean_post,  burn_in + 14),
  "zero_dyn_obs_err_at_h1":     _at(mean_zero,  burn_in),
  "zero_dyn_obs_err_at_h5":     _at(mean_zero,  burn_in + 4),
  "prior_rew_err_at_h5":        _at(mean_prior_rew, burn_in + 4),
  "prior_rew_err_at_h15":       _at(mean_prior_rew, burn_in + 14),
  "rew_baseline_at_h5":         _at(mean_pred_err, burn_in + 4),
}
print(json.dumps(summary, indent=2))